# LeetCode 2103 — Rings and Rods bằng Python

Link đề: https://leetcode.com/problems/rings-and-rods/ · Độ khó: **Easy** · Chủ đề: Hash Table, String, Bit Manipulation

```python
class Solution:
    def countPoints(self, rings: str) -> int:

```

## 1. Mục tiêu học

Sau bài này bạn sẽ:

1. Hiểu cách **đọc chuỗi theo cặp** `(màu, số cọc)` bằng bước nhảy `range(0, n, 2)` và lát cắt `s[i], s[i+1]`.
2. Biết dùng `dict` + `set` để gom dữ liệu theo nhóm (mỗi cọc có một tập màu).
3. Làm quen **bitmask**: mỗi màu là một bit, ba màu đủ khi mask bằng `0b111 = 7`.
4. So sánh 4 cách: quét brute force, dict-of-set, bitmask, tập các cặp `(cọc, màu)`.
5. Trace, test đối chiếu nhiều cách và tránh lỗi hay gặp (đếm vòng thay vì đếm màu, bỏ sót cọc).

## 2. Đề bài & ví dụ

Có **10 cái cọc** đánh số `0..9`. Có các vòng màu **R** (đỏ), **G** (xanh lá), **B** (xanh dương) lồng vào các cọc. Chuỗi `rings` dài `2n` mô tả `n` vòng: mỗi vòng gồm **2 ký tự liên tiếp**:

- ký tự ở vị trí chẵn: màu vòng (`'R'`, `'G'` hoặc `'B'`),
- ký tự ở vị trí lẻ ngay sau: số hiệu cọc (`'0'`..`'9'`) mà vòng đó được lồng vào.

Trả về **số cọc có đủ cả ba màu** R, G, B.

| Ví dụ | rings | Output |
|---|---|---|
| 1 | `"B0B6G0R6R0R6G9"` | `1` |
| 2 | `"B0R0G0R9R0B0G0"` | `1` |
| 3 | `"G4"` | `0` |

Ví dụ 1: cọc 0 có `B, G, R` (đủ); cọc 6 có `B, R, R` (thiếu G); cọc 9 chỉ có `G`. -> 1 cọc.

Constraints: `rings.length == 2 * n`, `1 <= n <= 100`; ký tự vị trí chẵn thuộc `{R, G, B}`; vị trí lẻ là chữ số `0..9`.

In [1]:
rings = "B0B6G0R6R0R6G9"
print("Độ dài:", len(rings), "-> số vòng n =", len(rings) // 2)
for i in range(0, len(rings), 2):
    print(f"vòng {i // 2}: màu={rings[i]} cọc={rings[i + 1]}")

Độ dài: 14 -> số vòng n = 7
vòng 0: màu=B cọc=0
vòng 1: màu=B cọc=6
vòng 2: màu=G cọc=0
vòng 3: màu=R cọc=6
vòng 4: màu=R cọc=0
vòng 5: màu=R cọc=6
vòng 6: màu=G cọc=9


## 3. Kiến thức nền

### 3.1. Duyệt chuỗi theo cặp bằng `range(0, n, 2)`

`range(start, stop, step)` với `step = 2` cho các chỉ số `0, 2, 4, ...`. Ở mỗi `i`, `s[i]` là ký tự đầu cặp và `s[i + 1]` là ký tự thứ hai.

In [2]:
s = "R3G5"
for i in range(0, len(s), 2):
    print(i, s[i], s[i + 1])

0 R 3
2 G 5


### 3.2. Lát cắt có bước: `s[0::2]` và `s[1::2]`

Cú pháp `s[a:b:c]`: lấy từ `a` đến trước `b`, bước `c`. `s[0::2]` = các ký tự ở vị trí chẵn (màu), `s[1::2]` = vị trí lẻ (cọc). `zip` ghép chúng lại thành các cặp.

In [3]:
rings = "B0B6G0R6R0R6G9"
colors, rods = rings[0::2], rings[1::2]
print(colors, rods)
print(list(zip(colors, rods)))

BBGRRRG 0606069
[('B', '0'), ('B', '6'), ('G', '0'), ('R', '6'), ('R', '0'), ('R', '6'), ('G', '9')]


### 3.3. `dict` + `set`: gom nhóm

Ta muốn "cọc nào có những màu nào": dùng `dict` có khóa là cọc, giá trị là `set` màu. `defaultdict(set)` tự tạo `set` rỗng khi khóa chưa có.

In [4]:
from collections import defaultdict

by_rod = defaultdict(set)
for color, rod in zip("BBGRRRG", "0603669"):
    by_rod[rod].add(color)
print(dict(by_rod))
print(len(by_rod["0"]))

{'0': {'G', 'B'}, '6': {'R', 'B'}, '3': {'R'}, '9': {'G'}}
2


### 3.4. Bitmask: mỗi màu là một bit

Mỗi màu ứng với một bit: `R = 1 (001)`, `G = 2 (010)`, `B = 4 (100)`. Toán tử `|` (OR) bật bit, `|=` cập nhật tại chỗ. Cả ba màu có mặt khi mask `== 7` (`0b111`). Ưu điểm: một số nguyên nhỏ thay cho cả `set`.

In [5]:
bit = {"R": 1, "G": 2, "B": 4}
mask = 0
for c in "BGR":
    mask |= bit[c]
    print(c, "->", bin(mask), mask)
print(mask == 7)

B -> 0b100 4
G -> 0b110 6
R -> 0b111 7
True


## 4. Ý tưởng nền tảng — tính bằng tay

`rings = "B0B6G0R6R0R6G9"`. Tách thành các cặp: `B0, B6, G0, R6, R0, R6, G9`.

| Cọc | Màu có |
|---|---|
| 0 | B, G, R (đủ) |
| 6 | B, R (thiếu G) |
| 9 | G |
| 1..5, 7, 8 | không có gì |

Đếm cọc đủ ba màu: **1**. Ý tưởng chung: với **mỗi cọc**, biết tập màu của nó, đếm xem có bao nhiêu cọc có tập màu = `{R, G, B}`. Chú ý: một màu có thể lặp nhiều lần trên cùng cọc (`R6` hai lần) nhưng chỉ tính **một** màu.

## 5. Cách 1 — Brute Force: với mỗi cọc, với mỗi màu, quét cả chuỗi

Duyệt 10 cọc. Với mỗi cọc và mỗi màu trong `"RGB"`, quét hết chuỗi để xem có cặp `(màu, cọc)` hay không. Cọc có cả ba màu thì cộng 1.

In [6]:
def count_points_brute(rings):
    ans = 0
    for rod in "0123456789":
        has_all = True
        for color in "RGB":
            found = False
            for i in range(0, len(rings), 2):
                if rings[i] == color and rings[i + 1] == rod:
                    found = True
                    break
            if not found:
                has_all = False
                break
        if has_all:
            ans += 1
    return ans

print(count_points_brute("B0B6G0R6R0R6G9"))   # 1
print(count_points_brute("B0R0G0R9R0B0G0"))   # 1
print(count_points_brute("G4"))               # 0

1
1
0


**Độ phức tạp:** `10 cọc × 3 màu × n vòng` = `O(30·n)`. Vì 10 và 3 là hằng số nên thực chất `O(n)`, rất nhanh với `n ≤ 100`. Tuy nhiên nó **quét lại chuỗi 30 lần** — nút thắt về mặt tư duy: ta nên đọc chuỗi **một lần** và ghi nhận kết quả vào bảng.

## 6. Ý tưởng bước đột phá — đọc một lần, ghi vào bảng theo cọc

Insight: mỗi vòng `(màu, cọc)` chỉ cho ta biết "cọc này có thêm màu kia". Vì vậy duyệt chuỗi **một lần** và ghi vào cấu trúc `rod -> tập màu`.

Quy tắc: **nếu** gặp vòng `(c, r)` **thì** thêm `c` vào tập của cọc `r`. Cuối cùng, **nếu** tập của cọc có đủ 3 màu **thì** cọc đó được tính.

Việc lặp lại cùng một màu trên cùng cọc không làm tập (hoặc mask) thay đổi -> đúng yêu cầu "có đủ ba màu", không phụ thuộc số vòng.

## 7. Các cách tiếp theo

### Cách 2 — `defaultdict(set)`: cọc -> tập màu

In [7]:
from collections import defaultdict

def count_points_dict_set(rings):
    colors_on = defaultdict(set)
    for i in range(0, len(rings), 2):
        colors_on[rings[i + 1]].add(rings[i])
    return sum(1 for s in colors_on.values() if len(s) == 3)

print(count_points_dict_set("B0B6G0R6R0R6G9"))
print(dict(defaultdict(set)))

1
{}


In [8]:
from collections import defaultdict

class SolutionDictSet:
    def countPoints(self, rings: str) -> int:
        colors_on = defaultdict(set)
        for i in range(0, len(rings), 2):
            colors_on[rings[i + 1]].add(rings[i])
        return sum(1 for s in colors_on.values() if len(s) == 3)

print(SolutionDictSet().countPoints("G4"))   # 0

0


Độ phức tạp Cách 2: `O(n)` thời gian, `O(10 × 3)` = `O(1)` bộ nhớ (tối đa 10 cọc, mỗi cọc ≤ 3 màu).

### Cách 3 — Bitmask 10 ô (cách gọn nhất)

Dùng mảng `mask[10]`. Mỗi vòng: `mask[cọc] |= bit[màu]`. Cuối cùng đếm các cọc có `mask == 7`.

In [9]:
def count_points_mask(rings):
    bit = {"R": 1, "G": 2, "B": 4}
    mask = [0] * 10
    for i in range(0, len(rings), 2):
        mask[int(rings[i + 1])] |= bit[rings[i]]
    return mask.count(7)

print(count_points_mask("B0B6G0R6R0R6G9"))
print(count_points_mask("B0R0G0R9R0B0G0"))

1
1


In [10]:
class SolutionMask:
    def countPoints(self, rings: str) -> int:
        bit = {"R": 1, "G": 2, "B": 4}
        mask = [0] * 10
        for i in range(0, len(rings), 2):
            mask[int(rings[i + 1])] |= bit[rings[i]]
        return mask.count(7)

print(SolutionMask().countPoints("G4"))

0


Độ phức tạp Cách 3: `O(n)` thời gian, `O(10)` bộ nhớ. `mask.count(7)` đếm số ô bằng 7.

### Cách 4 — Tập các cặp duy nhất + `Counter` theo cọc

Gom mọi cặp `(cọc, màu)` **khác nhau** vào một `set`; sau đó đếm số cặp theo cọc: cọc nào có đúng 3 cặp khác nhau thì đủ màu. Cách này dùng ý "khử trùng trước, đếm sau".

In [11]:
from collections import Counter

def count_points_pairs(rings):
    pairs = {(rings[i + 1], rings[i]) for i in range(0, len(rings), 2)}
    per_rod = Counter(rod for rod, _ in pairs)
    return sum(1 for v in per_rod.values() if v == 3)

print(sorted({(rings[i + 1], rings[i]) for i in range(0, len(rings), 2)}))
print(count_points_pairs("B0B6G0R6R0R6G9"))

[('0', 'B'), ('0', 'G'), ('0', 'R'), ('6', 'B'), ('6', 'R'), ('9', 'G')]
1


In [12]:
from collections import Counter

class SolutionPairs:
    def countPoints(self, rings: str) -> int:
        pairs = {(rings[i + 1], rings[i]) for i in range(0, len(rings), 2)}
        per_rod = Counter(rod for rod, _ in pairs)
        return sum(1 for v in per_rod.values() if v == 3)

print(SolutionPairs().countPoints("B0R0G0R9R0B0G0"))

1


Độ phức tạp Cách 4: `O(n)` thời gian, `O(n)` bộ nhớ trong trường hợp xấu (nhưng chặn trên 30 cặp vì chỉ có 10 cọc × 3 màu).

## 8. TRACE từng bước

Trace Cách 3 (bitmask) trên `"B0B6G0R6R0R6G9"`: bảng trạng thái `mask` theo từng vòng (chỉ in cọc bị thay đổi).

| Vòng | (màu, cọc) | mask của cọc sau khi cập nhật |
|---|---|---|
| 0 | (B, 0) | cọc 0: `100` |
| 1 | (B, 6) | cọc 6: `100` |
| 2 | (G, 0) | cọc 0: `110` |
| 3 | (R, 6) | cọc 6: `101` |
| 4 | (R, 0) | cọc 0: `111` (đủ) |
| 5 | (R, 6) | cọc 6: `101` (không đổi) |
| 6 | (G, 9) | cọc 9: `010` |

In [13]:
def trace_mask(rings):
    bit = {"R": 1, "G": 2, "B": 4}
    mask = [0] * 10
    for i in range(0, len(rings), 2):
        color, rod = rings[i], int(rings[i + 1])
        before = mask[rod]
        mask[rod] |= bit[color]
        print(f"vòng {i // 2}: ({color}, {rod})  cọc {rod}: {before:03b} -> {mask[rod]:03b}"
              + ("  <- ĐỦ 3 MÀU" if mask[rod] == 7 and before != 7 else ""))
    print("mask cuối:", mask)
    return mask.count(7)

print("Đáp án:", trace_mask("B0B6G0R6R0R6G9"))

vòng 0: (B, 0)  cọc 0: 000 -> 100
vòng 1: (B, 6)  cọc 6: 000 -> 100
vòng 2: (G, 0)  cọc 0: 100 -> 110
vòng 3: (R, 6)  cọc 6: 100 -> 101
vòng 4: (R, 0)  cọc 0: 110 -> 111  <- ĐỦ 3 MÀU
vòng 5: (R, 6)  cọc 6: 101 -> 101
vòng 6: (G, 9)  cọc 9: 000 -> 010
mask cuối: [7, 0, 0, 0, 0, 0, 5, 0, 0, 2]
Đáp án: 1


In [14]:
def trace_dict_set(rings):
    from collections import defaultdict
    d = defaultdict(set)
    for i in range(0, len(rings), 2):
        d[rings[i + 1]].add(rings[i])
        print(f"đọc {rings[i:i + 2]!r}: {dict(d)}")
    return sum(1 for s in d.values() if len(s) == 3)

print("Đáp án:", trace_dict_set("R1G1B1R2"))

đọc 'R1': {'1': {'R'}}
đọc 'G1': {'1': {'R', 'G'}}
đọc 'B1': {'1': {'R', 'G', 'B'}}
đọc 'R2': {'1': {'R', 'G', 'B'}, '2': {'R'}}
Đáp án: 1


## 9. Code nộp LeetCode

Dùng Cách 3 (bitmask).

In [15]:
from typing import List, Optional

class Solution:
    def countPoints(self, rings: str) -> int:
        bit = {"R": 1, "G": 2, "B": 4}
        mask = [0] * 10
        for i in range(0, len(rings), 2):
            mask[int(rings[i + 1])] |= bit[rings[i]]
        return mask.count(7)

print(Solution().countPoints("B0B6G0R6R0R6G9"))

1


## 10. Vì sao thuật toán đúng?

- **Invariant:** sau khi xử lý `k` vòng đầu, `mask[r]` là OR của các bit màu đã xuất hiện trên cọc `r` trong `k` vòng đó. Mỗi vòng chỉ có thể **bật thêm** bit, không bao giờ tắt, nên mask chỉ tăng.
- Cọc `r` thỏa đề **khi và chỉ khi** cả ba màu đã xuất hiện trên nó, tức cả ba bit đều bật, tức `mask[r] == 0b111 == 7`.
- Vòng lặp lại (cùng màu, cùng cọc) không đổi mask nên đúng ý "chỉ cần có mặt".
- Có đúng 10 cọc, `mask.count(7)` đếm hết mọi cọc đạt điều kiện.

Cách dict-of-set đúng với lý do tương tự: `set` tự loại màu trùng; `len(set) == 3` nghĩa là đủ R, G, B (vì chỉ có 3 màu có thể).

In [16]:
# Kiểm tra invariant: mask chỉ tăng và khớp với tập màu thực
import random
random.seed(103)
bit = {"R": 1, "G": 2, "B": 4}
for _ in range(500):
    n = random.randint(1, 12)
    rings = "".join(random.choice("RGB") + random.choice("0123") for _ in range(n))
    mask = [0] * 10
    colors = defaultdict(set)
    for i in range(0, len(rings), 2):
        r = int(rings[i + 1])
        old = mask[r]
        mask[r] |= bit[rings[i]]
        colors[r].add(rings[i])
        assert mask[r] >= old
        assert mask[r] == sum(bit[c] for c in colors[r])
print("invariant OK")

invariant OK


## 11. Phân tích Time / Space Complexity

`n` = số vòng = `len(rings) / 2`. Hằng số: 10 cọc, 3 màu.

| Cách | Time | Space |
|---|---|---|
| 1. Brute quét lại | `O(10·3·n)` = `O(n)` | `O(1)` |
| 2. `defaultdict(set)` | `O(n)` | `O(1)` (≤ 30 phần tử) |
| 3. Bitmask | `O(n)` | `O(1)` (10 số nguyên) |
| 4. Tập cặp + Counter | `O(n)` | `O(n)` tạm, tối đa 30 cặp thực tế |

Mọi cách đều tuyến tính; khác biệt chủ yếu ở độ gọn và hằng số.

## 12. Bộ test

Ví dụ đề, edge case (một vòng, chỉ một màu, mọi cọc đủ màu) và đối chiếu ngẫu nhiên 4 cách.

In [17]:
ALL = [count_points_brute, count_points_dict_set, count_points_mask, count_points_pairs]
cases = [
    ("B0B6G0R6R0R6G9", 1),
    ("B0R0G0R9R0B0G0", 1),
    ("G4", 0),
    ("R1G1B1", 1),
    ("R1G1", 0),
    ("R1R1R1", 0),
    ("R0G0B0R1G1B1R2G2B2R3G3B3R4G4B4R5G5B5R6G6B6R7G7B7R8G8B8R9G9B9", 10),
    ("R0G1B2", 0),
]
for r, exp in cases:
    for f in ALL:
        assert f(r) == exp, (f.__name__, r, exp)
    assert Solution().countPoints(r) == exp
print("Tất cả test ví dụ + edge case qua:", len(cases), "ca x", len(ALL) + 1, "cách")

Tất cả test ví dụ + edge case qua: 8 ca x 5 cách


In [18]:
import random
random.seed(2103)
for _ in range(3000):
    n = random.randint(1, 30)
    rings = "".join(random.choice("RGB") + random.choice("0123456789") for _ in range(n))
    base = count_points_brute(rings)
    assert count_points_dict_set(rings) == base
    assert count_points_mask(rings) == base
    assert count_points_pairs(rings) == base
    assert Solution().countPoints(rings) == base
print("3000 test ngẫu nhiên: các cách khớp nhau")

3000 test ngẫu nhiên: các cách khớp nhau


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Đếm số vòng thay vì đếm số màu khác nhau

Một cọc có 3 vòng đỏ chưa phải đủ ba màu.

In [19]:
def wrong_count_rings(rings):
    cnt = [0] * 10
    for i in range(0, len(rings), 2):
        cnt[int(rings[i + 1])] += 1
    return sum(1 for c in cnt if c >= 3)

print("sai:", wrong_count_rings("R1R1R1"), "| đúng:", Solution().countPoints("R1R1R1"))

sai: 1 | đúng: 0


Sai vì `R1R1R1` có 3 vòng nhưng chỉ một màu. Phải đếm **màu khác nhau** (`set`/`mask`), không đếm số vòng.

### Lỗi 2: Đọc sai thứ tự cặp (nhầm cọc với màu)

In [20]:
def wrong_order(rings):
    bit = {"R": 1, "G": 2, "B": 4}
    mask = [0] * 10
    for i in range(0, len(rings), 2):
        # nhầm: rings[i] là màu, không phải cọc
        try:
            mask[int(rings[i])] |= bit[rings[i + 1]]
        except ValueError as e:
            return "ValueError: " + str(e)
    return mask.count(7)

print(wrong_order("B0B6G0R6R0R6G9"))

ValueError: invalid literal for int() with base 10: 'B'


Ký tự vị trí chẵn là **màu** (`B`), vị trí lẻ là **cọc** (`0`). Đổi chỗ thì `int("B")` lỗi. Nhớ: `rings[i]` = màu, `rings[i + 1]` = cọc.

### Lỗi 3: Duyệt từng ký tự thay vì từng cặp (bước nhảy 1)

In [21]:
def wrong_step_one(rings):
    seen = defaultdict(set)
    for i in range(len(rings) - 1):                   # bước 1 -> ghép cả "0B", "6G"...
        seen[rings[i + 1]].add(rings[i])
    return sum(1 for s in seen.values() if len(s) == 3)

print("sai:", wrong_step_one("B0B6G0R6R0R6G9"), "| đúng:", Solution().countPoints("B0B6G0R6R0R6G9"))
print("Các khóa sai (lẫn chữ cái và chữ số):", sorted({k for i in range(len("B0G0")-1) for k in ["B0G0"[i+1]]}))

sai: 1 | đúng: 1
Các khóa sai (lẫn chữ cái và chữ số): ['0', 'G']


Bước nhảy phải là `2`. Với bước 1 ta ghép nhầm cả cặp `(cọc, màu)` đảo ngược, sinh tập lẫn chữ số và chữ cái.

### Lỗi 4 (thêm): Quên rằng cọc không xuất hiện thì mask là `0`

Nếu dùng `dict` chỉ có cọc xuất hiện thì ổn; nếu dùng mảng 10 ô thì nhớ `[0] * 10`, không phải `[0] * n`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "có đủ **tất cả** loại" (ba màu, 26 chữ cái, các số `1..n`) -> nghĩ **set** hoặc **bitmask**.
- Chuỗi mã hóa theo cặp / bộ ba cố định -> `range(0, n, k)` hoặc `zip(s[0::k], s[1::k])`.
- Số loại nhỏ (≤ 30) -> bitmask thay `set` để gọn và nhanh.
- Kiểm tra "đủ" bằng `mask == (1 << k) - 1` (k loại) — ở đây `k = 3` nên là 7.
- Tránh quét lại dữ liệu nhiều lần: gom vào bảng ngay lúc đọc.

## 15. Biến thể của bài

1. Có `k` màu thay vì 3 (mask `(1 << k) - 1`).
2. Đếm cọc có **ít nhất 2 màu** hoặc **đúng 1 màu**.
3. Trả về **danh sách** các cọc đủ màu, sắp xếp tăng dần.
4. Mỗi vòng có kèm **số lượng** (ví dụ `R3` nghĩa là 3 vòng đỏ) — khi đó cần tổng, không chỉ tồn tại.
5. Cọc có thể mang tên chuỗi (ví dụ `"A"`, `"B"`) thay vì chữ số: dùng `dict` thay mảng 10 ô.

In [22]:
def rods_with_all(rings):
    bit = {"R": 1, "G": 2, "B": 4}
    mask = [0] * 10
    for i in range(0, len(rings), 2):
        mask[int(rings[i + 1])] |= bit[rings[i]]
    return [r for r in range(10) if mask[r] == 7]

def count_at_least_two(rings):
    bit = {"R": 1, "G": 2, "B": 4}
    mask = [0] * 10
    for i in range(0, len(rings), 2):
        mask[int(rings[i + 1])] |= bit[rings[i]]
    return sum(1 for m in mask if bin(m).count("1") >= 2)

print(rods_with_all("B0B6G0R6R0R6G9"))
print(count_at_least_two("B0B6G0R6R0R6G9"))

[0]
2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute quét lại | Với mỗi (cọc, màu) quét chuỗi | `O(30n)` | `O(1)` | Học cơ bản, trực giác |
| 2. `defaultdict(set)` | Cọc -> tập màu | `O(n)` | `O(1)` | Dễ đọc, dễ mở rộng |
| 3. Bitmask | Mỗi màu là một bit | `O(n)` | `O(1)` | **Nên dùng**, gọn, nhanh |
| 4. Tập cặp + Counter | Khử trùng cặp rồi đếm theo cọc | `O(n)` | `O(1)`~`O(n)` | Khi cần tư duy "khử trùng" |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 — Đoán output:** `Solution().countPoints("R1G1B1R1")` bằng bao nhiêu?

**Bài 2 — Tính tay:** `"R2G2R3B3G3"`: cọc nào đủ ba màu?

**Bài 3 — Điền TODO:** hoàn thành `rod_colors(rings)` trả về `dict` cọc -> chuỗi màu đã sắp xếp (ví dụ `{'0': 'BGR'}` cho kết quả sorted).

**Bài 4 — Tự viết lại:** viết `exercise_count_points(rings)` bằng cách khác (ví dụ chỉ dùng `set` cặp).

**Bài 5 — Mini challenge:** viết `count_points_k(rings, colors)` với tập màu tùy ý `colors` (ví dụ `"RGBY"`), đếm cọc có đủ mọi màu trong `colors`.

Các hàm chỉ được định nghĩa (không gọi) để không gây lỗi khi chạy toàn bộ.

In [23]:
def rod_colors(rings):
    # TODO: trả dict {cọc: "".join(sorted(tập màu))}
    pass

# assert rod_colors("B0G0R0") == {"0": "BGR"}

In [24]:
def exercise_count_points(rings):
    # TODO: dùng set các cặp (cọc, màu) rồi đếm
    pass

# assert exercise_count_points("B0B6G0R6R0R6G9") == 1

In [25]:
def count_points_k(rings, colors):
    # TODO: tổng quát hóa với tập màu tùy ý
    pass

# assert count_points_k("R0G0B0Y0R1G1B1", "RGBY") == 1

## 18. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Chỉ cọc `1` có vòng | Cọc 1 có R, G, B | Đáp án `1` |
| 2 | Tách cặp `R2 G2 R3 B3 G3` | Cọc 2: R,G; cọc 3: B,G,... | Không cọc nào đủ -> `0` |
| 3 | `defaultdict(set)` | Duyệt `range(0, len, 2)` | `{r: "".join(sorted(s)) for r, s in d.items()}` |
| 4 | Tập `(rings[i+1], rings[i])` | `Counter` các cọc | Đếm cọc có 3 cặp |
| 5 | Mask theo vị trí trong `colors` | Đủ khi `mask == (1 << len(colors)) - 1` | Bỏ qua vòng màu ngoài `colors` |

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `1` (cọc 1 có R, G, B). **Bài 2:** cọc 2 có `R, G`; cọc 3 có `R, B, G` -> chỉ cọc 3 đủ ba màu -> đáp án `1`.

In [26]:
assert Solution().countPoints("R1G1B1R1") == 1
assert Solution().countPoints("R2G2R3B3G3") == 1      # cọc 3 đủ R,G,B

def rod_colors_answer(rings):
    d = defaultdict(set)
    for i in range(0, len(rings), 2):
        d[rings[i + 1]].add(rings[i])
    return {r: "".join(sorted(s)) for r, s in d.items()}

def exercise_count_points_answer(rings):
    pairs = {(rings[i + 1], rings[i]) for i in range(0, len(rings), 2)}
    return sum(1 for r in set(p[0] for p in pairs) if sum(1 for q in pairs if q[0] == r) == 3)

def count_points_k_answer(rings, colors):
    full = (1 << len(colors)) - 1
    idx = {c: i for i, c in enumerate(colors)}
    mask = defaultdict(int)
    for i in range(0, len(rings), 2):
        if rings[i] in idx:
            mask[rings[i + 1]] |= 1 << idx[rings[i]]
    return sum(1 for m in mask.values() if m == full)

assert rod_colors_answer("B0G0R0") == {"0": "BGR"}
assert exercise_count_points_answer("B0B6G0R6R0R6G9") == 1
assert count_points_k_answer("R0G0B0Y0R1G1B1", "RGBY") == 1
assert count_points_k_answer("R0G0B0R1G1B1", "RGB") == 2
print("Đáp án bài tập: tất cả assert đều qua")

Đáp án bài tập: tất cả assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template**

```python
class Solution:
    def countPoints(self, rings: str) -> int:
        bit = {"R": 1, "G": 2, "B": 4}
        mask = [0] * 10
        for i in range(0, len(rings), 2):      # mỗi vòng = 2 ký tự
            mask[int(rings[i + 1])] |= bit[rings[i]]
        return mask.count(7)                    # 0b111
```

**Checklist**

- [ ] Đọc theo **cặp**: `rings[i]` = màu, `rings[i + 1]` = cọc, bước `2`.
- [ ] Đếm **màu khác nhau** trên mỗi cọc, không đếm số vòng.
- [ ] Mảng 10 ô cho 10 cọc (`0..9`), khởi tạo `0`.
- [ ] Đủ ba màu <=> `mask == 7`.
- [ ] Đã test `"G4"`, `"R1R1R1"`, các ví dụ của đề và so sánh ngẫu nhiên với brute force.